```mermaid
graph TD;
    A-->B;
    A-->C;
    B-->D;
    C-->D;
```

```mermaid
classDiagram
    %% Factory Method
    class Product
    class Creator {
        +createProduct() Product
    }
    class ConcreteProduct
    class ConcreteCreator
    Creator <|-- ConcreteCreator
    Product <|-- ConcreteProduct
    ConcreteCreator --> Product : create()

In [22]:
import json
import subprocess

# Edge浏览器的路径
edge_path = r'C:\Program Files (x86)\Microsoft\Edge\Application\msedge.exe'

file_path = 'test.json'
head = 'https://www.kuaishou.com/profile/'

result_path = 'result.txt'

In [ ]:
links = []

with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)
for item in data["fols"]:
    links.append(head + item["user_id"])
    
with open(result_path, 'w', encoding='utf-8') as f:
    f.writelines("\n".join(links))
for link in links:
    subprocess.run([edge_path, link])
    # print(link)


---
### 開幹

In [ ]:
link = ['https://www.kuaishou.com/profile/3xt3rwhxxgwb6xw']


In [ ]:
from playwright.sync_api import sync_playwright
import time
import json
import subprocess

def scrape_kuaishou_videos(profile_url):
    video_urls = []

    with sync_playwright() as p:
        browser = p.chromium.launch(headless=False)  # 可改成 True 隐藏浏览器
        page = browser.new_page()
        page.goto(profile_url)

        # 滚动加载视频
        for _ in range(10):  # 根据视频数量调整滚动次数
            page.mouse.wheel(0, 2000)
            time.sleep(1)

        # 抓取视频链接
        elements = page.query_selector_all("a[href*='/short-video/']")
        for el in elements:
            href = el.get_attribute("href")
            if href and href.startswith("/short-video/"):
                full_url = "https://www.kuaishou.com" + href
                if full_url not in video_urls:
                    video_urls.append(full_url)

        browser.close()

    return video_urls


def download_kuaishou_videos(urls):
    for url in urls:
        print(f"Downloading: {url}")
        cmd = ["yt-dlp", "-o", "%(title)s.%(ext)s", url]
        subprocess.run(cmd)


if __name__ == "__main__":
    profile = "https://www.kuaishou.com/profile/3xt3rwhxxgwb6xw"
    videos = scrape_kuaishou_videos(profile)

    # 保存视频链接
    with open("kuaishou_videos.json", "w", encoding="utf-8") as f:
        json.dump(videos, f, ensure_ascii=False, indent=2)

    print(f"共获取到 {len(videos)} 个视频")

    # 下载视频
    download_kuaishou_videos(videos)
